## Imports

In [1]:
from pathlib import Path
import sys
import os

import numpy as np

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule

In [2]:
from common import read_file_str, show_formatted_cpp, replace_constants_in_kernel

In [3]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Parameters

In [4]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

In [5]:
from src.parameters import STRESS_THRESHOLD, STRESS_WEIGHTING, VERTEX_RESOLUTION

## Load in data from mesh

In [6]:
vertices = np.load('./profiling/numpy/piece_vertices.npy')
acceleration = np.load('./profiling/numpy/piece_acceleration.npy')
stress_relations = np.load('./profiling/numpy/piece_stress_relations.npy')

In [7]:
acceleration *= 0

## Cuda Parameters

In [8]:
BLOCK_SIZE = 1024
NR_BLOCKS = (len(acceleration) + BLOCK_SIZE - 1) // BLOCK_SIZE

## Compile cuda kernel

In [9]:
cuda_code = """
__device__ __inline__ float3 subtract(float3 a, float3 b) {
    return make_float3(b.x - a.x, b.y - a.y, b.z - a.z);
}

__device__ __inline__ float normL2(float3 v) {
    return sqrtf(v.x * v.x + v.y * v.y + v.z * v.z);
}

__device__ __inline__ void scaleVector(float3 &v, float s) {
    v.x *= s;
    v.y *= s;
    v.z *= s;
}

__global__ void apply_stress(float *acceleration, float *vertices,
                             unsigned int* stress_relations, int nr_stress_relations) {
    int pair_idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (pair_idx >= nr_stress_relations) return;

    unsigned int from_ind = stress_relations[pair_idx * 2];
    unsigned int to_ind = stress_relations[pair_idx * 2 + 1];

    float3 from_vertex = make_float3(
        vertices[from_ind * 3],
        vertices[from_ind * 3 + 1],
        vertices[from_ind * 3 + 2]
    );
    float3 to_vertex = make_float3(
        vertices[to_ind * 3],
        vertices[to_ind * 3 + 1],
        vertices[to_ind * 3 + 2]
    );

    float3 vector = subtract(from_vertex, to_vertex);
    float distance = normL2(vector);
    float stress_amount = distance / STRESS_RESTING_AMOUNT;
    if (stress_amount > 1 + STRESS_THRESHOLD) {
        scaleVector(vector, STRESS_WEIGHTING);
        acceleration[from_ind * 3] += vector.x;
        acceleration[from_ind * 3 + 1] += vector.y;
        acceleration[from_ind * 3 + 2] += vector.z;
        acceleration[to_ind * 3] -= vector.x;
        acceleration[to_ind * 3 + 1] -= vector.y;
        acceleration[to_ind * 3 + 2] -= vector.z;
    } else if (stress_amount < 1 - STRESS_THRESHOLD) {
        scaleVector(vector, STRESS_WEIGHTING);
        acceleration[from_ind * 3] -= vector.x;
        acceleration[from_ind * 3 + 1] -= vector.y;
        acceleration[from_ind * 3 + 2] -= vector.z;
        acceleration[to_ind * 3] += vector.x;
        acceleration[to_ind * 3 + 1] += vector.y;
        acceleration[to_ind * 3 + 2] += vector.z;
    }
}

"""

In [10]:
parameter_updates = {
    "STRESS_THRESHOLD": STRESS_THRESHOLD,
    "STRESS_WEIGHTING": STRESS_WEIGHTING,
    "STRESS_RESTING_AMOUNT": VERTEX_RESOLUTION
}

In [11]:
cuda_code = replace_constants_in_kernel(cuda_code, parameter_updates)

In [12]:
show_formatted_cpp(cuda_code)

In [13]:
mod = SourceModule(cuda_code)

C:\Users\CYBORG\AppData\Local\Temp\ipykernel_10024\3464942708.py:1: UserWarning: The CUDA compiler succeeded, but said the following:
kernel.cu

  mod = SourceModule(cuda_code)


## Set-up memory for running kernel

In [14]:
apply_stress = mod.get_function("apply_stress")

In [15]:
nr_stress_relations = np.int32(len(stress_relations))

### Allocate memory to gpu

In [16]:
assert acceleration.flatten().flags['C_CONTIGUOUS']
assert vertices.flatten().flags['C_CONTIGUOUS']
assert stress_relations.flatten().flags['C_CONTIGUOUS']

In [17]:
acceleration_gpu = cuda.mem_alloc(acceleration.nbytes)
vertices_gpu = cuda.mem_alloc(vertices.nbytes)
stress_relations_gpu = cuda.mem_alloc(stress_relations.nbytes)

In [18]:
cuda.memcpy_htod(acceleration_gpu, acceleration.flatten())
cuda.memcpy_htod(vertices_gpu, vertices.flatten())
cuda.memcpy_htod(stress_relations_gpu, stress_relations.flatten())

## Create function

In [19]:
def apply_stress_kernel():
    apply_stress(acceleration_gpu, vertices_gpu, stress_relations_gpu, nr_stress_relations,
                 block=(BLOCK_SIZE, 1, 1), grid=(NR_BLOCKS, 1, 1))

## Check output is as expected

In [20]:
apply_stress_kernel()

In [21]:
cuda.memcpy_dtoh(vertices, vertices_gpu)
cuda.memcpy_dtoh(acceleration, acceleration_gpu)
cuda.memcpy_dtoh(stress_relations, stress_relations_gpu)

In [22]:
expected_after_stress = np.load('./profiling/numpy/expected_piece_acceleration_after_stress.npy')

In [23]:
expected_after_stress.max()

2121.015

In [24]:
acceleration.max()

20.382893

## Profile function

In [25]:
%timeit apply_stress_kernel()

14.2 µs ± 1.5 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
